In [12]:
crmp_old_url = "postgresql://crmp@db.pcic.uvic.ca/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/climo-data-importer/.pgpass"
metnorth_old_url = "postgresql://metnorth@db.pcic.uvic.ca:5433/metnorth?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/climo-data-importer/.pgpass"
crmp_new_url = "postgresql+psycopg2://crmp@/crmp?host=proddb01.pcic.uvic.ca,proddb02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/climo-data-importer/.pgpass"
metnorth_new_url = "postgresql+psycopg2://metnorth_ro@/metnorth?host=proddb01.pcic.uvic.ca,proddb02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/climo-data-importer/.pgpass"

In [13]:
import pandas as pd
from sqlalchemy import create_engine, text

engine_old = create_engine(crmp_old_url)
engine_new = create_engine(crmp_new_url)


## Overall Counts in each DB for obs_raw

In [14]:
with engine_old.connect() as conn:
    old_count = conn.execute(text("SELECT COUNT(*) FROM obs_raw")).scalar()

with engine_new.connect() as conn:
    new_count = conn.execute(text("SELECT COUNT(*) FROM obs_raw")).scalar()

print(f"crmp_old obs_raw count: {old_count:,}")
print(f"crmp     obs_raw count: {new_count:,}")
print(f"Difference:             {new_count - old_count:+,}")


crmp_old obs_raw count: 1,104,776,013
crmp     obs_raw count: 1,104,776,998
Difference:             +985


## Counts broken down by network

In [15]:
network_query = text("""
    SELECT
        mn.network_name,
        COUNT(o.obs_raw_id) AS obs_count
    FROM obs_raw o
    JOIN meta_history mh ON o.history_id = mh.history_id
    JOIN meta_station ms ON mh.station_id = ms.station_id
    JOIN meta_network mn ON ms.network_id = mn.network_id
    GROUP BY mn.network_name
    ORDER BY mn.network_name
""")

with engine_old.connect() as conn:
    df_old = pd.read_sql(network_query, conn).rename(columns={"obs_count": "count_old"})

with engine_new.connect() as conn:
    df_new = pd.read_sql(network_query, conn).rename(columns={"obs_count": "count_new"})

by_network = df_old.merge(df_new, on="network_name", how="outer").fillna(0)
by_network[["count_old", "count_new"]] = by_network[["count_old", "count_new"]].astype(int)
by_network["difference"] = by_network["count_new"] - by_network["count_old"]
by_network = by_network.sort_values("difference")

print(f"Networks with differences: {(by_network['difference'] != 0).sum()} of {len(by_network)}")
by_network[by_network["difference"] != 0]


Networks with differences: 3 of 23


,network_name,count_old,count_new,difference
6,CRD,31903275,31903232,-43
11,ENV-ASP,12475408,12475435,27
13,FLNRO-WMB,356071952,356072953,1001


## Counts broken down by station

In [8]:
station_query = text("""
    SELECT
        mh.history_id,
        TRIM(mh.station_name) AS station_name,
        COUNT(o.obs_raw_id) AS obs_count
    FROM obs_raw o
    JOIN meta_history mh ON o.history_id = mh.history_id
    GROUP BY  mh.history_id, TRIM(mh.station_name)
    ORDER BY  TRIM(mh.station_name)
""")

with engine_old.connect() as conn:
    df_old = pd.read_sql(station_query, conn).rename(columns={"obs_count": "count_old"})

with engine_new.connect() as conn:
    df_new = pd.read_sql(station_query, conn).rename(columns={"obs_count": "count_new"})

by_station = df_old.merge(
    df_new,
    on=["history_id", "station_name"],
    how="outer",
).fillna(0)
by_station[["count_old", "count_new"]] = by_station[["count_old", "count_new"]].astype(int)
by_station["difference"] = by_station["count_new"] - by_station["count_old"]
by_station_diff = by_station[by_station["difference"] != 0].sort_values("difference")

print(f"Stations with differences: {len(by_station_diff)} of {len(by_station)}")
by_station_diff


Stations with differences: 937 of 9475


,history_id,station_name,count_old,count_new,difference
2232,2863,Floe Lk,466606,435261,-31345
8760,14410,Colpitti Creek,499065,467721,-31344
2251,2882,Mission Ridge,431172,399828,-31344
9013,15023,Kaza Lake,421987,390651,-31336
9046,15128,Dickson Lake,232469,201133,-31336
...,...,...,...,...,...
2200,2830,Mount Pondosy,117813,117716,-97
2199,2829,Tahtsa Lake,115012,114938,-74
2198,2828,Mount Wells,111113,111061,-52
2197,2827,Tahtsa Intake,141242,141214,-28


## Counts broken down by variable


In [5]:
variable_query = text("""
    SELECT
        mn.network_name,
        mv.standard_name,
        mv.cell_method,
        mv.unit,
        COUNT(o.obs_raw_id) AS obs_count
    FROM obs_raw o
    JOIN meta_vars mv ON o.vars_id = mv.vars_id
    JOIN meta_network mn ON mv.network_id = mn.network_id
    GROUP BY mn.network_name, mv.standard_name, mv.cell_method, mv.unit
    ORDER BY mn.network_name, mv.standard_name
""")

with engine_old.connect() as conn:
    df_old = pd.read_sql(variable_query, conn).rename(columns={"obs_count": "count_old"})

with engine_new.connect() as conn:
    df_new = pd.read_sql(variable_query, conn).rename(columns={"obs_count": "count_new"})

by_variable = df_old.merge(
    df_new,
    on=["network_name", "standard_name", "cell_method", "unit"],
    how="outer",
).fillna(0)
by_variable[["count_old", "count_new"]] = by_variable[["count_old", "count_new"]].astype(int)
by_variable["difference"] = by_variable["count_new"] - by_variable["count_old"]
by_variable_diff = by_variable[by_variable["difference"] != 0].sort_values("difference")

print(f"Variables with differences: {len(by_variable_diff)} of {len(by_variable)}")
by_variable_diff


Variables with differences: 83 of 266


,network_name,standard_name,cell_method,unit,count_old,count_new,difference
32,BCH,air_temperature,time: point,celsius,20114331,19196731,-917600
153,FLNRO-WMB,air_temperature,time: point,celsius,66771616,66225816,-545800
164,FLNRO-WMB,wind_speed,time: mean,km/h,66613571,66067993,-545578
36,BCH,liquid_water_content_of_surface_snow,time: point,mm,6740458,6295365,-445093
45,BCH,surface_snow_thickness,time: point,cm,5465809,5061672,-404137
...,...,...,...,...,...,...,...
249,RT,wind_speed,time: mean (interval: 10 months),degrees,579,531,-48
245,RT,lwe_thickness_of_surface_snow_amount,time: point,mm,33019,32984,-35
244,RT,lwe_thickness_of_precipitation_amount,time: sum (interval: 1 hour),mm,51132,51104,-28
239,RT,lwe_thickness_of_precipitation,time: sum (interval: 3 hour),mm,50013,49987,-26


## Find gaps in Floe Lk:
<!-- 
	history_id	station_name	count_old	count_new	difference
    2232	2863	Floe Lk	466606	435261	-31345 
>

In [9]:
floe_lk_query = text("""
    SELECT
        o.obs_time
    FROM obs_raw o
    JOIN meta_history mh ON o.history_id = mh.history_id
    WHERE TRIM(mh.station_name) = 'Floe Lk'
    ORDER BY o.obs_time
""")

with engine_old.connect() as conn:
    floe_old = pd.read_sql(floe_lk_query, conn)
    floe_old["obs_time"] = pd.to_datetime(floe_old["obs_time"])

with engine_new.connect() as conn:
    floe_new = pd.read_sql(floe_lk_query, conn)
    floe_new["obs_time"] = pd.to_datetime(floe_new["obs_time"])

# Timestamps in old but missing from new
missing_in_new = floe_old[~floe_old["obs_time"].isin(floe_new["obs_time"])]
# Timestamps in new but missing from old
missing_in_old = floe_new[~floe_new["obs_time"].isin(floe_old["obs_time"])]

print(f"Old DB: {len(floe_old):,} rows")
print(f"New DB: {len(floe_new):,} rows")
print(f"In old, missing from new: {len(missing_in_new):,}")
print(f"In new, missing from old: {len(missing_in_old):,}")

# Show the date ranges of the gaps
print("\n--- Missing from new (date range) ---")
if len(missing_in_new):
    print(f"  {missing_in_new['obs_time'].min()} → {missing_in_new['obs_time'].max()}")
    # Show by year to spot pattern
    missing_in_new.groupby(missing_in_new["obs_time"].dt.year).size().rename("missing_count").to_frame()


Old DB: 466,606 rows
New DB: 435,261 rows
In old, missing from new: 31,345
In new, missing from old: 0

--- Missing from new (date range) ---
  2025-12-27 14:00:00 → 2026-04-14 12:15:00


In [10]:
missing_in_new.groupby(missing_in_new["obs_time"].dt.year).size().rename("missing_count").to_frame()


,missing_count
obs_time,
2025,1272
2026,30073


In [11]:
# Find gaps > 2 hours in the old DB for Floe Lk
floe_old_sorted = floe_old.sort_values("obs_time").reset_index(drop=True)
floe_old_sorted["gap"] = floe_old_sorted["obs_time"].diff()

gaps = floe_old_sorted[floe_old_sorted["gap"] > pd.Timedelta(hours=2)].copy()
gaps["gap_start"] = floe_old_sorted["obs_time"].shift(1)[gaps.index]
gaps["gap_end"] = gaps["obs_time"]
gaps["gap_hours"] = gaps["gap"].dt.total_seconds() / 3600

print(f"Gaps > 2 hours in old DB: {len(gaps)}")
gaps[["gap_start", "gap_end", "gap_hours"]].sort_values("gap_hours", ascending=False)


Gaps > 2 hours in old DB: 7167


,gap_start,gap_end,gap_hours
17467,2008-12-01 00:00:00,2017-06-01 00:00:00,74496.000000
363188,2025-02-18 13:15:00,2025-04-01 00:00:00,994.750000
19933,2019-09-30 00:00:00,2019-11-01 00:00:00,768.000000
19657,2019-05-31 00:00:00,2019-07-01 00:00:00,744.000000
435261,2025-12-09 13:15:00,2025-12-27 14:00:00,432.750000
...,...,...,...
7739,2000-01-31 00:00:00,2000-01-31 23:59:59,23.999722
147339,2023-01-14 00:00:00,2023-01-14 14:00:00,14.000000
363203,2025-04-05 00:00:00,2025-04-05 13:00:00,13.000000
21493,2021-04-03 00:00:00,2021-04-03 12:55:00,12.916667


In [12]:
# Check the same gaps > 2 hours in the new DB
floe_new_sorted = floe_new.sort_values("obs_time").reset_index(drop=True)
floe_new_sorted["gap"] = floe_new_sorted["obs_time"].diff()

gaps_new = floe_new_sorted[floe_new_sorted["gap"] > pd.Timedelta(hours=2)].copy()
gaps_new["gap_start"] = floe_new_sorted["obs_time"].shift(1)[gaps_new.index]
gaps_new["gap_end"] = gaps_new["obs_time"]
gaps_new["gap_hours"] = gaps_new["gap"].dt.total_seconds() / 3600

print(f"Gaps > 2 hours in new DB: {len(gaps_new)}")
gaps_new[["gap_start", "gap_end", "gap_hours"]].sort_values("gap_hours", ascending=False)


Gaps > 2 hours in new DB: 7166


,gap_start,gap_end,gap_hours
17467,2008-12-01 00:00:00,2017-06-01 00:00:00,74496.000000
363188,2025-02-18 13:15:00,2025-04-01 00:00:00,994.750000
19933,2019-09-30 00:00:00,2019-11-01 00:00:00,768.000000
19657,2019-05-31 00:00:00,2019-07-01 00:00:00,744.000000
14402,2006-02-13 00:00:00,2006-02-14 00:00:00,24.000000
...,...,...,...
8021,2000-04-30 00:00:00,2000-04-30 23:59:59,23.999722
147339,2023-01-14 00:00:00,2023-01-14 14:00:00,14.000000
363203,2025-04-05 00:00:00,2025-04-05 13:00:00,13.000000
21493,2021-04-03 00:00:00,2021-04-03 12:55:00,12.916667


In [13]:
# Compare gaps side by side: merge on gap_start to see which gaps differ between DBs
gaps_old = gaps[["gap_start", "gap_end", "gap_hours"]].rename(columns={"gap_end": "gap_end_old", "gap_hours": "gap_hours_old"})
gaps_new_cmp = gaps_new[["gap_start", "gap_end", "gap_hours"]].rename(columns={"gap_end": "gap_end_new", "gap_hours": "gap_hours_new"})

gaps_cmp = gaps_old.merge(gaps_new_cmp, on="gap_start", how="outer", indicator=True)

# Gaps only in old DB (not in new)
only_in_old = gaps_cmp[gaps_cmp["_merge"] == "left_only"][["gap_start", "gap_end_old", "gap_hours_old"]]
# Gaps only in new DB (not in old)
only_in_new = gaps_cmp[gaps_cmp["_merge"] == "right_only"][["gap_start", "gap_end_new", "gap_hours_new"]]
# Gaps in both but with different end times or durations
in_both = gaps_cmp[gaps_cmp["_merge"] == "both"].copy()
in_both["end_diff_hours"] = (in_both["gap_end_new"] - in_both["gap_end_old"]).dt.total_seconds() / 3600
duration_differs = in_both[in_both["end_diff_hours"] != 0]

print(f"Gaps only in old DB: {len(only_in_old)}")
print(f"Gaps only in new DB: {len(only_in_new)}")
print(f"Gaps in both with different end time: {len(duration_differs)}")

print("\n--- Gaps only in old DB ---")
display(only_in_old.sort_values("gap_hours_old", ascending=False).head(20))

print("\n--- Gaps only in new DB ---")
display(only_in_new.sort_values("gap_hours_new", ascending=False).head(20))

print("\n--- Gaps in both but end time differs ---")
display(duration_differs[["gap_start", "gap_end_old", "gap_end_new", "gap_hours_old", "gap_hours_new", "end_diff_hours"]].sort_values("end_diff_hours").head(20))


Gaps only in old DB: 1
Gaps only in new DB: 0
Gaps in both with different end time: 0

--- Gaps only in old DB ---


,gap_start,gap_end_old,gap_hours_old
7166,2025-12-09 13:15:00,2025-12-27 14:00:00,432.75



--- Gaps only in new DB ---


,gap_start,gap_end_new,gap_hours_new



--- Gaps in both but end time differs ---


,gap_start,gap_end_old,gap_end_new,gap_hours_old,gap_hours_new,end_diff_hours
